# Customer Churn Prediction

## Portfolio Project Summary

**Domain:** Customer Analytics  
**Project Type:** Classification  
**Repository Link:** https://github.com/sundaralingam48/sundaralingam48.github.io/tree/main/projects/03_customer_churn_prediction

### Business Problem
Companies lose revenue when customers leave. This project identifies customers who are more likely to churn so a business can take early retention actions.

### Project Summary
This project predicts customer churn using tenure, monthly charges, satisfaction, support tickets, logins, and product usage.

### Skills Demonstrated
- Data cleaning and preparation
- Exploratory data analysis
- Feature engineering
- Model training and evaluation
- Business interpretation and recommendations
- Ethical consideration of data science results

> This notebook is self-contained and uses simulated data so it can run successfully in an employer-facing portfolio without requiring private or restricted data. The same workflow can be adapted to a real dataset.


## 1. Import Libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance

pd.set_option("display.max_columns", 50)


## 2. Create or Load Dataset

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(103)
n = 800

df = pd.DataFrame({
    'tenure_months': np.random.normal(50, 15, n),
    'monthly_charges': np.random.normal(50, 15, n),
    'support_tickets': np.random.normal(50, 15, n),
    'satisfaction_score': np.random.normal(50, 15, n),
    'logins_30d': np.random.normal(50, 15, n),
    'num_products': np.random.normal(50, 15, n)
})

# Keep numeric values practical for modeling
for col in df.columns:
    df[col] = np.maximum(df[col], 0)

# Create a target with a meaningful signal plus noise
signal = (
    0.06 * df['tenure_months']
    - 0.04 * df['monthly_charges']
    + 0.05 * df['support_tickets']
    + 0.03 * df['logins_30d']
    + np.random.normal(0, 6, n)
)

threshold = np.percentile(signal, 60)
df['churned'] = (signal > threshold).astype(int)

df.head()

## 3. Exploratory Data Analysis

In [ ]:
print("Dataset shape:", df.shape)
display(df.describe().T)

target_col = 'churned'
print("\nTarget distribution / summary:")
display(df[target_col].describe())


In [ ]:
# Visualization 1: target distribution
target_col = 'churned'

plt.figure(figsize=(8, 5))
if df[target_col].nunique() <= 10:
    df[target_col].value_counts().sort_index().plot(kind="bar")
    plt.ylabel("Count")
else:
    df[target_col].plot(kind="hist", bins=30)
    plt.ylabel("Frequency")
plt.title(f"Distribution of {target_col}")
plt.xlabel(target_col)
plt.tight_layout()
plt.show()


In [ ]:
# Visualization 2: correlation with target
target_col = 'churned'
corr = df.corr(numeric_only=True)[target_col].drop(target_col).sort_values()

plt.figure(figsize=(8, 5))
corr.plot(kind="barh")
plt.title(f"Feature Correlation with {target_col}")
plt.xlabel("Correlation")
plt.tight_layout()
plt.show()


## 4. Model Training and Evaluation

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

features = ['tenure_months', 'monthly_charges', 'support_tickets', 'satisfaction_score', 'logins_30d', 'num_products']
target = 'churned'

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000))
    ]),
    "Random Forest": RandomForestClassifier(n_estimators=150, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

results = []
trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    trained_models[name] = model
    preds = model.predict(X_test)
    results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, preds),
        "precision": precision_score(y_test, preds, zero_division=0),
        "recall": recall_score(y_test, preds, zero_division=0),
        "f1": f1_score(y_test, preds, zero_division=0)
    })

results_df = pd.DataFrame(results).sort_values("f1", ascending=False)
display(results_df)

best_model_name = results_df.iloc[0]["model"]
best_model = trained_models[best_model_name]
print("Best model:", best_model_name)

best_preds = best_model.predict(X_test)
print("\nClassification Report:")
print(classification_report(y_test, best_preds))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, best_preds))


## 5. Model Visualization

In [ ]:
best_preds = best_model.predict(X_test)
cm = confusion_matrix(y_test, best_preds)

plt.figure(figsize=(5, 4))
plt.imshow(cm)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center")
plt.tight_layout()
plt.show()


## 6. Feature Importance

In [ ]:
# Feature importance using permutation importance
perm = permutation_importance(best_model, X_test, y_test, n_repeats=10, random_state=42)
importance_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance": perm.importances_mean
}).sort_values("importance", ascending=True)

plt.figure(figsize=(8, 5))
plt.barh(importance_df["feature"], importance_df["importance"])
plt.title("Permutation Feature Importance")
plt.xlabel("Mean Importance")
plt.tight_layout()
plt.show()

display(importance_df.sort_values("importance", ascending=False))


## Business Interpretation and Recommendations

### Key Findings
The notebook compares multiple models and selects the best-performing model based on the most appropriate evaluation metric. The feature importance section identifies which variables most strongly influence the prediction.

### Recommendations
1. Use the model as a decision-support tool, not as the only decision-maker.
2. Monitor model performance over time as new data becomes available.
3. Review high-impact features for fairness, quality, and possible bias.
4. Build a simple dashboard or reporting layer so stakeholders can understand results.

### Ethical Considerations
This project should avoid using sensitive personal information unless it is necessary, allowed, and properly protected. Model results should be explainable to business users, and decisions should be reviewed for fairness and unintended impact.

### Future Improvements
- Replace simulated data with a real validated dataset.
- Add cross-validation and hyperparameter tuning.
- Store model outputs in a reusable report.
- Deploy the workflow as a dashboard, API, or scheduled analytics process.
